In [44]:
import re
import pandas as pd

def split_into_sentences(text):
    """
    Baseline JD segmentation.

    Preserves markdown/bullet lines and avoids splitting
    common abbreviations such as Ltd., Inc., etc.
    """

    if not isinstance(text, str):
        return []

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # First split on markdown/bullet lines.
    lines = text.split("\n")

    segments = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Remove markdown bullet markers only for processing.
        line = re.sub(r"^[-*]\s+", "", line)

        # Headings / labels should remain intact.
        if (
            line.startswith("#")
            or line.endswith(":")
            or re.match(r"^\*\*.*\*\*:?$", line)
        ):
            segments.append(line)
            continue

        # Protect common abbreviations.
        protected = re.sub(
            r"\b(?:Ltd|Inc|Mr|Mrs|Ms|Dr|Prof|e\.g|i\.e)\.",
            lambda m: m.group(0).replace(".", "<DOT>"),
            line,
            flags=re.IGNORECASE
        )

        # Split genuine sentence boundaries.
        parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", protected)

        for part in parts:
            part = part.replace("<DOT>", ".").strip()

            if part:
                segments.append(part)

    return segments

In [45]:
import pandas as pd

clean_df = pd.read_csv("../data/netsol_clean.csv")

print("Rows:", len(clean_df))
print("Columns:", clean_df.columns.tolist())

Rows: 1910
Columns: ['source_file', 'candidate_id', 'resume', 'job_description', 'requirement', 'label']


In [46]:
test_jd = """
Requirements:
- 3+ years of Python development experience.
- Experience with FastAPI and Docker.
- Bachelor's degree in Computer Science.
- Kubernetes experience is preferred.
- Strong communication and teamwork skills.
"""

sentences = split_into_sentences(test_jd)

for i, sentence in enumerate(sentences):
    print(i, repr(sentence))

0 'Requirements:'
1 '3+ years of Python development experience.'
2 'Experience with FastAPI and Docker.'
3 "Bachelor's degree in Computer Science."
4 'Kubernetes experience is preferred.'
5 'Strong communication and teamwork skills.'


In [47]:
REQUIRED_PATTERNS = [
    r"\brequired\b",
    r"\bmust\b",
    r"\bmust have\b",
    r"\bmandatory\b",
    r"\bessential\b",
]

PREFERRED_PATTERNS = [
    r"\bpreferred\b",
    r"\bpreferable\b",
    r"\bnice to have\b",
    r"\bplus\b",
    r"\bdesirable\b",
]

In [48]:
def detect_importance(text):
    text_lower = text.lower()

    for pattern in REQUIRED_PATTERNS:
        if re.search(pattern, text_lower):
            return "required"

    for pattern in PREFERRED_PATTERNS:
        if re.search(pattern, text_lower):
            return "preferred"

    return "unspecified"

In [49]:
for sentence in sentences[1:]:
    print(sentence)
    print("Importance:", detect_importance(sentence))
    print()

3+ years of Python development experience.
Importance: unspecified

Experience with FastAPI and Docker.
Importance: unspecified

Bachelor's degree in Computer Science.
Importance: unspecified

Kubernetes experience is preferred.
Importance: preferred

Strong communication and teamwork skills.
Importance: unspecified



In [50]:
REQUIREMENT_PATTERNS = [
    r"\bexperience\b",
    r"\bskills?\b",
    r"\bproficien(?:cy|t)\b",
    r"\bknowledge\b",
    r"\bunderstanding\b",
    r"\bability\b",
    r"\bdegree\b",
    r"\bbachelor'?s\b",
    r"\bmaster'?s\b",
    r"\bcertification\b",
    r"\bcertified\b",
    r"\byears?\b",
    r"\brequired\b",
    r"\bmust\b",
    r"\bpreferred\b",
]

In [51]:
def is_requirement(text):
    text_lower = text.lower()

    return any(
        re.search(pattern, text_lower)
        for pattern in REQUIREMENT_PATTERNS
    )

In [52]:
for sentence in sentences:
    print(
        f"{is_requirement(sentence):5} | {sentence}"
    )

    0 | Requirements:
    1 | 3+ years of Python development experience.
    1 | Experience with FastAPI and Docker.
    1 | Bachelor's degree in Computer Science.
    1 | Kubernetes experience is preferred.
    1 | Strong communication and teamwork skills.


In [53]:
TYPE_PATTERNS = {
    "education": [
        r"\bbachelor'?s\b",
        r"\bmaster'?s\b",
        r"\bph\.?d\.?\b",
        r"\bdegree\b",
        r"\bdiploma\b",
    ],

    "experience": [
        r"\b\d+\+?\s*(?:years?|yrs?)\b",
        r"\byears? of experience\b",
        r"\bexperience\b",
    ],

    "certification": [
        r"\bcertification\b",
        r"\bcertified\b",
        r"\blicen[cs]e\b",
    ],

    "programming_language": [
        r"\bpython\b",
        r"\bjava\b",
        r"\bc\+\+\b",
        r"\bc#\b",
        r"\bjavascript\b",
        r"\btypescript\b",
        r"\bruby\b",
        r"\bgo\b",
        r"\brust\b",
    ],

    "soft_skill": [
        r"\bcommunication\b",
        r"\bteamwork\b",
        r"\bleadership\b",
        r"\binterpersonal\b",
        r"\bproblem[- ]solving\b",
        r"\bcollaboration\b",
    ],

    "technical_skill": [
        r"\bmachine learning\b",
        r"\bdeep learning\b",
        r"\bdata analysis\b",
        r"\bdata engineering\b",
        r"\bsoftware development\b",
        r"\bcloud computing\b",
        r"\bcybersecurity\b",
    ],
}

In [54]:
def classify_requirement_type(text):
    text_lower = text.lower()

    # More specific categories first
    priority_order = [
        "education",
        "certification",
        "experience",
        "programming_language",
        "soft_skill",
        "technical_skill",
    ]

    for requirement_type in priority_order:
        patterns = TYPE_PATTERNS[requirement_type]

        if any(re.search(pattern, text_lower) for pattern in patterns):
            return requirement_type

    return "other"

In [55]:
for sentence in sentences[1:]:
    print(
        f"{classify_requirement_type(sentence):25} | {sentence}"
    )

experience                | 3+ years of Python development experience.
experience                | Experience with FastAPI and Docker.
education                 | Bachelor's degree in Computer Science.
experience                | Kubernetes experience is preferred.
soft_skill                | Strong communication and teamwork skills.


In [85]:
def extract_experience_range(text):
    text_lower = text.lower()

    # Example: 3-5 years
    range_match = re.search(
        r'(\d+)\s*[-–]\s*(\d+)\s*(?:years?|yrs?)',
        text_lower
    )

    if range_match:
        return {
            "minimum_years": int(range_match.group(1)),
            "maximum_years": int(range_match.group(2))
        }

    # Example: 3+ years / at least 3 years
    minimum_match = re.search(
        r'(?:at least\s+)?(\d+)\+?\s*(?:years?|yrs?)',
        text_lower
    )

    if minimum_match:
        return {
            "minimum_years": int(minimum_match.group(1)),
            "maximum_years": None
        }

    return {
        "minimum_years": None,
        "maximum_years": None
    }

In [86]:
experience_examples = [
    "3-5 years of experience in business development.",
    "3+ years of Python development experience.",
    "At least 5 years of software engineering experience.",
    "Experience with Python."
]

for text in experience_examples:
    print(text)
    print(extract_experience_range(text))
    print()

3-5 years of experience in business development.
{'minimum_years': 3, 'maximum_years': 5}

3+ years of Python development experience.
{'minimum_years': 3, 'maximum_years': None}

At least 5 years of software engineering experience.
{'minimum_years': 5, 'maximum_years': None}

Experience with Python.
{'minimum_years': None, 'maximum_years': None}



In [57]:
for sentence in sentences[1:]:
    print(
        sentence,
        "→",
        extract_minimum_years(sentence)
    )

3+ years of Python development experience. → 3
Experience with FastAPI and Docker. → None
Bachelor's degree in Computer Science. → None
Kubernetes experience is preferred. → None
Strong communication and teamwork skills. → None


In [58]:
KNOWN_SKILLS = [
    "Python",
    "Java",
    "C++",
    "JavaScript",
    "TypeScript",
    "FastAPI",
    "Django",
    "React",
    "Docker",
    "Kubernetes",
    "AWS",
    "Azure",
    "GCP",
    "SQL",
    "PostgreSQL",
    "MongoDB",
    "TensorFlow",
    "PyTorch",
    "Machine Learning",
    "Deep Learning",
    "Data Analysis",
    "Microsoft Excel",
    "Salesforce",
    "Power BI",
]

In [59]:
def extract_skills(text):
    text_lower = text.lower()
    found = []

    for skill in KNOWN_SKILLS:
        pattern = r'(?<!\w)' + re.escape(skill.lower()) + r'(?!\w)'

        if re.search(pattern, text_lower):
            found.append(skill)

    return found

In [60]:
for sentence in sentences[1:]:
    print(
        sentence,
        "→",
        extract_skills(sentence)
    )

3+ years of Python development experience. → ['Python']
Experience with FastAPI and Docker. → ['FastAPI', 'Docker']
Bachelor's degree in Computer Science. → []
Kubernetes experience is preferred. → ['Kubernetes']
Strong communication and teamwork skills. → []


In [61]:
def extract_requirement(text):
    return {
        "text": text,
        "type": classify_requirement_type(text),
        "importance": detect_importance(text),
        "operator": detect_operator(text),
        "skills": extract_skills(text),
        "minimum_years": extract_minimum_years(text),
    }

In [62]:
for sentence in sentences[1:]:
    print(extract_requirement(sentence))
    print()

{'text': '3+ years of Python development experience.', 'type': 'experience', 'importance': 'unspecified', 'operator': None, 'skills': ['Python'], 'minimum_years': 3}

{'text': 'Experience with FastAPI and Docker.', 'type': 'experience', 'importance': 'unspecified', 'operator': 'AND', 'skills': ['FastAPI', 'Docker'], 'minimum_years': None}

{'text': "Bachelor's degree in Computer Science.", 'type': 'education', 'importance': 'unspecified', 'operator': None, 'skills': [], 'minimum_years': None}

{'text': 'Kubernetes experience is preferred.', 'type': 'experience', 'importance': 'preferred', 'operator': None, 'skills': ['Kubernetes'], 'minimum_years': None}

{'text': 'Strong communication and teamwork skills.', 'type': 'soft_skill', 'importance': 'unspecified', 'operator': 'AND', 'skills': [], 'minimum_years': None}



In [63]:
def split_compound_requirement(text):
    parts = re.split(
        r'\s+(?:and|or|&)\s+',
        text,
        flags=re.IGNORECASE
    )

    return [
        part.strip(" -.,")
        for part in parts
        if part.strip(" -.,")
    ]

In [64]:
compound_examples = [
    "Experience with FastAPI and Docker.",
    "Strong communication and teamwork skills.",
    "Python or Java experience.",
]

for text in compound_examples:
    print(text)
    print(split_compound_requirement(text))
    print()

Experience with FastAPI and Docker.
['Experience with FastAPI', 'Docker']

Strong communication and teamwork skills.
['Strong communication', 'teamwork skills']

Python or Java experience.
['Python', 'Java experience']



In [65]:
def detect_operator(text):
    text_lower = text.lower()

    if re.search(r"\s+or\s+", text_lower):
        return "OR"

    if re.search(r"\s+(?:and|&)\s+", text_lower):
        return "AND"

    return None

In [66]:
compound_examples = [
    "Experience with FastAPI and Docker.",
    "Strong communication and teamwork skills.",
    "Python or Java experience.",
    "Experience with Python.",
]

for text in compound_examples:
    print(text)
    print("Operator:", detect_operator(text))
    print()

Experience with FastAPI and Docker.
Operator: AND

Strong communication and teamwork skills.
Operator: AND

Python or Java experience.
Operator: OR

Experience with Python.
Operator: None



In [67]:
SKILL_TYPES = {
    "Python": "programming_language",
    "Java": "programming_language",
    "C++": "programming_language",
    "JavaScript": "programming_language",
    "TypeScript": "programming_language",

    "FastAPI": "framework",
    "Django": "framework",
    "React": "framework",
    "PyTorch": "framework",
    "TensorFlow": "framework",

    "Docker": "tool",
    "Kubernetes": "tool",
    "AWS": "tool",
    "Azure": "tool",
    "GCP": "tool",
    "Salesforce": "tool",
    "Power BI": "tool",

    "SQL": "technical_skill",
    "PostgreSQL": "technical_skill",
    "MongoDB": "technical_skill",
    "Machine Learning": "technical_skill",
    "Deep Learning": "technical_skill",
    "Data Analysis": "technical_skill",

    "Communication": "soft_skill",
    "Teamwork": "soft_skill",
    "Leadership": "soft_skill",
}

In [68]:
def classify_skill(skill):
    return SKILL_TYPES.get(skill, "other")

In [69]:
def extract_typed_skills(text):
    skills = extract_skills(text)

    return [
        {
            "name": skill,
            "type": classify_skill(skill)
        }
        for skill in skills
    ]

In [70]:
for sentence in sentences[1:]:
    print(sentence)
    print(extract_typed_skills(sentence))
    print()

3+ years of Python development experience.
[{'name': 'Python', 'type': 'programming_language'}]

Experience with FastAPI and Docker.
[{'name': 'FastAPI', 'type': 'framework'}, {'name': 'Docker', 'type': 'tool'}]

Bachelor's degree in Computer Science.
[]

Kubernetes experience is preferred.
[{'name': 'Kubernetes', 'type': 'tool'}]

Strong communication and teamwork skills.
[]



In [71]:
SKILL_ALIASES = {
    "Python programming": "Python",
    "Python development": "Python",
    "Python 3": "Python",
    "Python 3.x": "Python",

    "Postgres": "PostgreSQL",
    "Postgres SQL": "PostgreSQL",
    "PostgreSQL DB": "PostgreSQL",

    "React.js": "React",
    "ReactJS": "React",

    "Amazon Web Services": "AWS",

    "Google Cloud Platform": "GCP",
    "Google Cloud": "GCP",

    "Microsoft Azure": "Azure",

    "PowerBI": "Power BI",

    "K8s": "Kubernetes",
}

In [72]:
def normalize_skill(skill):
    return SKILL_ALIASES.get(skill, skill)

In [73]:
test_aliases = [
    "Python programming",
    "Python 3",
    "Postgres",
    "React.js",
    "Amazon Web Services",
    "K8s",
    "Docker",
]

for skill in test_aliases:
    print(f"{skill} → {normalize_skill(skill)}")

Python programming → Python
Python 3 → Python
Postgres → PostgreSQL
React.js → React
Amazon Web Services → AWS
K8s → Kubernetes
Docker → Docker


In [74]:
def normalize_requirement(text):
    raw_skills = extract_skills(text)

    normalized_skills = []

    for skill in raw_skills:
        canonical = normalize_skill(skill)

        normalized_skills.append({
            "name": canonical,
            "type": classify_skill(canonical)
        })

    return {
        "text": text,
        "type": classify_requirement_type(text),
        "importance": detect_importance(text),
        "operator": detect_operator(text),
        "skills": normalized_skills,
        "minimum_years": extract_minimum_years(text)
    }

In [75]:
for sentence in sentences[1:]:
    print(normalize_requirement(sentence))
    print()

{'text': '3+ years of Python development experience.', 'type': 'experience', 'importance': 'unspecified', 'operator': None, 'skills': [{'name': 'Python', 'type': 'programming_language'}], 'minimum_years': 3}

{'text': 'Experience with FastAPI and Docker.', 'type': 'experience', 'importance': 'unspecified', 'operator': 'AND', 'skills': [{'name': 'FastAPI', 'type': 'framework'}, {'name': 'Docker', 'type': 'tool'}], 'minimum_years': None}

{'text': "Bachelor's degree in Computer Science.", 'type': 'education', 'importance': 'unspecified', 'operator': None, 'skills': [], 'minimum_years': None}

{'text': 'Kubernetes experience is preferred.', 'type': 'experience', 'importance': 'preferred', 'operator': None, 'skills': [{'name': 'Kubernetes', 'type': 'tool'}], 'minimum_years': None}

{'text': 'Strong communication and teamwork skills.', 'type': 'soft_skill', 'importance': 'unspecified', 'operator': 'AND', 'skills': [], 'minimum_years': None}



In [76]:
def extract_education(text):
    text_lower = text.lower()

    degree = None

    if re.search(r"\bbachelor'?s\b", text_lower):
        degree = "Bachelor's"
    elif re.search(r"\bmaster'?s\b", text_lower):
        degree = "Master's"
    elif re.search(r"\bph\.?d\.?\b", text_lower):
        degree = "PhD"
    elif re.search(r"\bdiploma\b", text_lower):
        degree = "Diploma"

    return degree

In [77]:
education_examples = [
    "Bachelor's degree in Computer Science.",
    "Master's degree in Data Science.",
    "PhD in Artificial Intelligence.",
    "Experience with Python."
]

for text in education_examples:
    print(text, "→", extract_education(text))

Bachelor's degree in Computer Science. → Bachelor's
Master's degree in Data Science. → Master's
PhD in Artificial Intelligence. → PhD
Experience with Python. → None


In [78]:
EDUCATION_FIELDS = [
    "Computer Science",
    "Computer Engineering",
    "Information Technology",
    "Data Science",
    "Artificial Intelligence",
    "Machine Learning",
    "Business Administration",
    "Marketing",
    "Mathematics",
    "Statistics",
]

In [79]:
def extract_education_fields(text):
    text_lower = text.lower()

    return [
        field
        for field in EDUCATION_FIELDS
        if field.lower() in text_lower
    ]

In [80]:
for text in education_examples:
    print(
        text,
        "→",
        extract_education_fields(text)
    )

Bachelor's degree in Computer Science. → ['Computer Science']
Master's degree in Data Science. → ['Data Science']
PhD in Artificial Intelligence. → ['Artificial Intelligence']
Experience with Python. → []


In [88]:
def normalize_requirement(text):
    raw_skills = extract_skills(text)

    normalized_skills = []

    for skill in raw_skills:
        canonical = normalize_skill(skill)

        normalized_skills.append({
            "name": canonical,
            "type": classify_skill(canonical)
        })

    return {
        "text": text,
        "type": classify_requirement_type(text),
        "importance": detect_importance(text),
        "operator": detect_operator(text),
        "skills": normalized_skills,
        "experience": extract_experience_range(text),
        "education": {
            "degree": extract_education(text),
            "fields": extract_education_fields(text)
        }
    }

In [89]:
sample_jd = clean_df["job_description"].drop_duplicates().iloc[0]

print(sample_jd)

### Business Development Executive

**Company:** Innovate Dynamics Ltd.

**Location:** Hyderabad, India

**Role Overview:**
Join Innovate Dynamics Ltd. as a Business Development Executive, where you will play a pivotal role in driving business growth by generating leads and managing key client accounts. Leverage your expertise in economic analysis and marketing strategies to enhance our market presence and build strong client relationships.

**Responsibilities:**
- Develop and execute comprehensive business development plans to achieve company objectives.
- Identify and engage prospective clients across diverse niches, including software development and outsourcing.
- Manage and grow relationships with key accounts to drive long-term success.
- Utilize digital marketing strategies to boost client engagement and retention.
- Collaborate with cross-functional teams to develop tailored client solutions.
- Utilize LinkedIn Sales Navigator to identify and approach new business leads.

**Qua

In [90]:
sample_sentences = split_into_sentences(sample_jd)

for i, sentence in enumerate(sample_sentences):
    print(i, repr(sentence))

0 '### Business Development Executive'
1 '**Company:** Innovate Dynamics Ltd.'
2 '**Location:** Hyderabad, India'
3 '**Role Overview:**'
4 'Join Innovate Dynamics Ltd. as a Business Development Executive, where you will play a pivotal role in driving business growth by generating leads and managing key client accounts.'
5 'Leverage your expertise in economic analysis and marketing strategies to enhance our market presence and build strong client relationships.'
6 '**Responsibilities:**'
7 'Develop and execute comprehensive business development plans to achieve company objectives.'
8 'Identify and engage prospective clients across diverse niches, including software development and outsourcing.'
9 'Manage and grow relationships with key accounts to drive long-term success.'
10 'Utilize digital marketing strategies to boost client engagement and retention.'
11 'Collaborate with cross-functional teams to develop tailored client solutions.'
12 'Utilize LinkedIn Sales Navigator to identify a

In [92]:
for sentence in sample_sentences:
    if is_requirement(sentence):
        print(normalize_requirement(sentence))
        print()

{'text': 'Bachelor’s degree in Business Administration, Economics, or related field.', 'type': 'education', 'importance': 'unspecified', 'operator': 'OR', 'skills': [], 'experience': {'minimum_years': None, 'maximum_years': None}, 'education': {'degree': None, 'fields': ['Business Administration']}}

{'text': '3-5 years of experience in business development or related fields.', 'type': 'experience', 'importance': 'unspecified', 'operator': 'OR', 'skills': [], 'experience': {'minimum_years': 3, 'maximum_years': 5}, 'education': {'degree': None, 'fields': []}}

{'text': 'Strong proficiency with MS Office Suite and data analysis tools like Stata.', 'type': 'technical_skill', 'importance': 'unspecified', 'operator': 'AND', 'skills': [{'name': 'Data Analysis', 'type': 'technical_skill'}], 'experience': {'minimum_years': None, 'maximum_years': None}, 'education': {'degree': None, 'fields': []}}

{'text': 'Excellent verbal and written communication skills.', 'type': 'soft_skill', 'importance'

In [93]:
unique_jds = clean_df["job_description"].drop_duplicates().reset_index(drop=True)

print("Unique JDs:", len(unique_jds))

for i, jd in enumerate(unique_jds):
    print(f"\n{'=' * 80}")
    print(f"JD {i}")
    print("=" * 80)
    print(jd[:1000])

Unique JDs: 18

JD 0
### Business Development Executive

**Company:** Innovate Dynamics Ltd.

**Location:** Hyderabad, India

**Role Overview:**
Join Innovate Dynamics Ltd. as a Business Development Executive, where you will play a pivotal role in driving business growth by generating leads and managing key client accounts. Leverage your expertise in economic analysis and marketing strategies to enhance our market presence and build strong client relationships.

**Responsibilities:**
- Develop and execute comprehensive business development plans to achieve company objectives.
- Identify and engage prospective clients across diverse niches, including software development and outsourcing.
- Manage and grow relationships with key accounts to drive long-term success.
- Utilize digital marketing strategies to boost client engagement and retention.
- Collaborate with cross-functional teams to develop tailored client solutions.
- Utilize LinkedIn Sales Navigator to identify and approach new b

In [94]:
annotation_columns = [
    "example_id",
    "jd_id",
    "text",
    "gold_is_requirement",
    "gold_type",
    "gold_importance",
    "gold_components",
    "gold_min_years",
    "gold_max_years",
]

gold_pilot = pd.DataFrame(columns=annotation_columns)

print(gold_pilot)

Empty DataFrame
Columns: [example_id, jd_id, text, gold_is_requirement, gold_type, gold_importance, gold_components, gold_min_years, gold_max_years]
Index: []


In [95]:
GOLD_TYPES = [
    "technical_skill",
    "tool",
    "programming_language",
    "framework",
    "domain_knowledge",
    "experience",
    "education",
    "certification",
    "soft_skill",
    "responsibility",
    "capability",
    "other",
]

GOLD_IMPORTANCE = [
    "required",
    "preferred",
    "unspecified",
]

In [96]:
candidate_requirements = []

for jd_id, jd in enumerate(unique_jds):

    sentences = split_into_sentences(jd)

    for sentence in sentences:

        if is_requirement(sentence):

            candidate_requirements.append({
                "jd_id": jd_id,
                "text": sentence
            })

candidate_requirements_df = pd.DataFrame(candidate_requirements)

print("Candidate requirement sentences:",
      len(candidate_requirements_df))

candidate_requirements_df.head(30)


Candidate requirement sentences: 94


,jd_id,text
0,0,"Bachelor’s degree in Business Administration, ..."
1,0,3-5 years of experience in business developmen...
2,0,Strong proficiency with MS Office Suite and da...
3,0,Excellent verbal and written communication ski...
4,0,Ability to multitask and manage multiple proje...
5,1,### Desired Skills and Experience:
6,1,"Strong programming skills in C++, HTML, and CSS."
7,1,"Proficiency in digital marketing, email market..."
8,1,Experience in managing large data sets and usi...
9,1,Excellent communication skills and ability to ...


In [97]:
print(
    candidate_requirements_df["jd_id"]
    .value_counts()
    .sort_index()
)

jd_id
0     5
1     7
2     5
3     5
4     5
5     3
6     7
7     4
8     1
9     8
10    6
11    2
12    4
13    2
14    9
15    7
16    9
17    5
Name: count, dtype: int64


In [98]:
candidate_requirements_df = candidate_requirements_df.reset_index(drop=True)

candidate_requirements_df.insert(
    0,
    "example_id",
    range(len(candidate_requirements_df))
)

pd.set_option("display.max_colwidth", 200)

candidate_requirements_df[
    ["example_id", "jd_id", "text"]
].to_string(index=False)

" example_id  jd_id                                                                                                                                                                                           text\n          0      0                                                                                                                     Bachelor’s degree in Business Administration, Economics, or related field.\n          1      0                                                                                                                             3-5 years of experience in business development or related fields.\n          2      0                                                                                                                    Strong proficiency with MS Office Suite and data analysis tools like Stata.\n          3      0                                                                                                                                         

In [99]:
candidate_requirements_df[
    ["example_id", "jd_id", "text"]
]

,example_id,jd_id,text
0,0,0,"Bachelor’s degree in Business Administration, Economics, or related field."
1,1,0,3-5 years of experience in business development or related fields.
2,2,0,Strong proficiency with MS Office Suite and data analysis tools like Stata.
3,3,0,Excellent verbal and written communication skills.
4,4,0,Ability to multitask and manage multiple projects simultaneously.
...,...,...,...
89,89,17,**Key Skills**:
90,90,17,Advanced electronic circuit and PCB design skills.
91,91,17,Excellent communication and interpersonal skills essential for strategic collaboration.
92,92,17,Solid understanding of embedded systems and applications.


In [104]:
selected_ids = [
    # JD 0
    0, 1, 2,

    # JD 1
    6, 8, 9,

    # JD 2
    13, 14, 15,

    # JD 3
    19, 20, 21,

    # JD 4
    24, 25, 26,

    # JD 5
    27, 28,

    # JD 6
    31, 33, 35, 36,

    # JD 7
    38, 39, 40,

    # JD 8
    41,

    # JD 9
    44, 45, 46, 49,

    # JD 10
    51, 53, 54,

    # JD 11
    56, 57,

    # JD 12
    60, 61,

    # JD 13
    63,

    # JD 14
    66, 67, 69,

    # JD 15
    75, 76,

    # JD 16
    81, 82, 84,

    # JD 17
    90, 92,

    # False-positive heading examples
    5, 12, 18, 23, 30
]

# Remove two redundant examples
selected_ids.remove(20)
selected_ids.remove(28)

print("Selected:", len(selected_ids))
print("Unique:", len(set(selected_ids)))

Selected: 50
Unique: 50


In [105]:
gold_pilot = candidate_requirements_df[
    candidate_requirements_df["example_id"].isin(selected_ids)
].copy()

gold_pilot = (
    gold_pilot[
        ["example_id", "jd_id", "text"]
    ]
    .sort_values("example_id")
    .reset_index(drop=True)
)

print("Rows:", len(gold_pilot))
print("Unique IDs:", gold_pilot["example_id"].nunique())

Rows: 50
Unique IDs: 50


In [106]:
print("Rows:", len(gold_pilot))
print("Unique IDs:", gold_pilot["example_id"].nunique())

print("\nJD distribution:")
print(gold_pilot["jd_id"].value_counts().sort_index())

Rows: 50
Unique IDs: 50

JD distribution:
jd_id
0     3
1     4
2     4
3     3
4     4
5     1
6     5
7     3
8     1
9     4
10    3
11    2
12    2
13    1
14    3
15    2
16    3
17    2
Name: count, dtype: int64


In [107]:
pd.set_option("display.max_colwidth", 200)

gold_pilot[
    ["example_id", "jd_id", "text"]
]

,example_id,jd_id,text
0,0,0,"Bachelor’s degree in Business Administration, Economics, or related field."
1,1,0,3-5 years of experience in business development or related fields.
2,2,0,Strong proficiency with MS Office Suite and data analysis tools like Stata.
3,5,1,### Desired Skills and Experience:
4,6,1,"Strong programming skills in C++, HTML, and CSS."
5,8,1,"Experience in managing large data sets and using BI tools (Power BI, Excel, Tableau)."
6,9,1,Excellent communication skills and ability to develop strong client relationships.
7,12,2,**Preferred Skills:**
8,13,2,Proficiency in strategic planning and market analysis.
9,14,2,Strong leadership skills with proven experience in team development.


In [108]:
GOLD_TYPES = [
    "technical_skill",
    "tool",
    "programming_language",
    "framework",
    "domain_knowledge",
    "experience",
    "education",
    "certification",
    "soft_skill",
    "responsibility",
    "capability",
    "other",
]

GOLD_IMPORTANCE = [
    "required",
    "preferred",
    "unspecified",
]

gold_pilot["gold_is_requirement"] = None
gold_pilot["gold_type"] = None
gold_pilot["gold_importance"] = None
gold_pilot["gold_components"] = None
gold_pilot["gold_min_years"] = None
gold_pilot["gold_max_years"] = None

gold_pilot = gold_pilot[
    [
        "example_id",
        "jd_id",
        "text",
        "gold_is_requirement",
        "gold_type",
        "gold_importance",
        "gold_components",
        "gold_min_years",
        "gold_max_years",
    ]
]

gold_pilot.head()

,example_id,jd_id,text,gold_is_requirement,gold_type,gold_importance,gold_components,gold_min_years,gold_max_years
0,0,0,"Bachelor’s degree in Business Administration, Economics, or related field.",None,None,None,None,None,None
1,1,0,3-5 years of experience in business development or related fields.,None,None,None,None,None,None
2,2,0,Strong proficiency with MS Office Suite and data analysis tools like Stata.,None,None,None,None,None,None
3,5,1,### Desired Skills and Experience:,None,None,None,None,None,None
4,6,1,"Strong programming skills in C++, HTML, and CSS.",None,None,None,None,None,None


In [110]:
gold_annotations = {
    0: {
        "gold_is_requirement": True,
        "gold_type": "education",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Business Administration", "type": "education_field"},
            {"name": "Economics", "type": "education_field"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    1: {
        "gold_is_requirement": True,
        "gold_type": "experience",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Business Development", "type": "domain_experience"},
        ],
        "gold_min_years": 3,
        "gold_max_years": 5,
    },

    2: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "required",
        "gold_components": [
            {"name": "MS Office Suite", "type": "tool"},
            {"name": "Data Analysis", "type": "technical_skill"},
            {"name": "Stata", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    3: {
        "gold_is_requirement": False,
        "gold_type": "other",
        "gold_importance": "unspecified",
        "gold_components": [],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    4: {
        "gold_is_requirement": True,
        "gold_type": "programming_language",
        "gold_importance": "required",
        "gold_components": [
            {"name": "C++", "type": "programming_language"},
            {"name": "HTML", "type": "programming_language"},
            {"name": "CSS", "type": "programming_language"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },
}

In [111]:
gold_annotations.update({

    # example_id 5
    5: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Large-scale data management", "type": "technical_skill"},
            {"name": "Power BI", "type": "tool"},
            {"name": "Excel", "type": "tool"},
            {"name": "Tableau", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 6
    6: {
        "gold_is_requirement": True,
        "gold_type": "soft_skill",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Communication", "type": "soft_skill"},
            {"name": "Client Relationship Development", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 7
    7: {
        "gold_is_requirement": False,
        "gold_type": "other",
        "gold_importance": "unspecified",
        "gold_components": [],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 8
    8: {
        "gold_is_requirement": True,
        "gold_type": "domain_knowledge",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "Strategic Planning", "type": "capability"},
            {"name": "Market Analysis", "type": "domain_knowledge"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 9
    9: {
        "gold_is_requirement": True,
        "gold_type": "soft_skill",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "Leadership", "type": "soft_skill"},
            {"name": "Team Development", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 10
    10: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "CRM Software", "type": "tool"},
            {"name": "Digital Marketing", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 11
    11: {
        "gold_is_requirement": False,
        "gold_type": "other",
        "gold_importance": "unspecified",
        "gold_components": [],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 12
    12: {
        "gold_is_requirement": True,
        "gold_type": "soft_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Communication", "type": "soft_skill"},
            {"name": "Interpersonal Skills", "type": "soft_skill"},
            {"name": "Client Engagement", "type": "capability"},
            {"name": "Collaboration", "type": "soft_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 13
    13: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Microsoft Office", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 14
    14: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "CRM Software", "type": "tool"},
            {"name": "Client Relationship Management", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 15
    15: {
        "gold_is_requirement": False,
        "gold_type": "other",
        "gold_importance": "unspecified",
        "gold_components": [],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 16
    16: {
        "gold_is_requirement": True,
        "gold_type": "experience",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "Sales Management", "type": "domain_experience"},
            {"name": "Digital Marketing", "type": "technical_skill"},
        ],
        "gold_min_years": 5,
        "gold_max_years": None,
    },

    # 17
    17: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Strategic Goal Setting", "type": "capability"},
            {"name": "Problem Solving", "type": "soft_skill"},
            {"name": "Communication", "type": "soft_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 18
    18: {
        "gold_is_requirement": True,
        "gold_type": "responsibility",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Team Leadership", "type": "capability"},
            {"name": "Team Supervision", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 19
    19: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Sales", "type": "domain_knowledge"},
            {"name": "Marketing", "type": "technical_skill"},
            {"name": "Analytical Skills", "type": "capability"},
            {"name": "SEO", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 20
    20: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Client Interaction", "type": "capability"},
            {"name": "Lead Conversion", "type": "capability"},
            {"name": "Marketing", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 21
    21: {
        "gold_is_requirement": False,
        "gold_type": "other",
        "gold_importance": "unspecified",
        "gold_components": [],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 22
    22: {
        "gold_is_requirement": True,
        "gold_type": "soft_skill",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Leadership", "type": "soft_skill"},
            {"name": "Stakeholder Management", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 23
    23: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Salesforce", "type": "tool"},
            {"name": "Vistex", "type": "tool"},
            {"name": "SAP ERP", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 24
    24: {
        "gold_is_requirement": True,
        "gold_type": "education",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Business Administration", "type": "education_field"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 25
    25: {
        "gold_is_requirement": True,
        "gold_type": "experience",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Sales Management", "type": "domain_experience"},
            {"name": "Business Development", "type": "domain_experience"},
            {"name": "Key Account Management", "type": "domain_experience"},
        ],
        "gold_min_years": 5,
        "gold_max_years": None,
    },

    # 26
    26: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "Power BI", "type": "tool"},
            {"name": "Oracle DV", "type": "tool"},
            {"name": "Tableau", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 27
    27: {
        "gold_is_requirement": True,
        "gold_type": "domain_knowledge",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "Agile", "type": "methodology"},
            {"name": "SCRUM", "type": "methodology"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 28
    28: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "Technical Team Leadership", "type": "capability"},
            {"name": "Stakeholder Communication", "type": "soft_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 29
    29: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Analytical Skills", "type": "capability"},
            {"name": "Opportunity Identification", "type": "capability"},
            {"name": "Market Trend Analysis", "type": "domain_knowledge"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },
})

In [112]:
print("Annotations added:", len(gold_annotations))

missing = [
    example_id
    for example_id in gold_pilot["example_id"]
    if example_id not in gold_annotations
]

print("Missing IDs:", missing)

Annotations added: 30
Missing IDs: [30, 31, 33, 35, 36, 38, 39, 40, 41, 44, 45, 46, 49, 51, 53, 54, 56, 57, 60, 61, 63, 66, 67, 69, 75, 76, 81, 82, 84, 90, 92]


In [113]:
print("Selected examples:", len(selected_ids))
print("Annotated examples:", len(gold_annotations))

missing_ids = sorted(
    set(selected_ids) - set(gold_annotations.keys())
)

print("Missing count:", len(missing_ids))
print("Missing IDs:", missing_ids)

Selected examples: 50
Annotated examples: 30
Missing count: 31
Missing IDs: [30, 31, 33, 35, 36, 38, 39, 40, 41, 44, 45, 46, 49, 51, 53, 54, 56, 57, 60, 61, 63, 66, 67, 69, 75, 76, 81, 82, 84, 90, 92]


In [115]:
gold_annotations.update({

    # 30 — JD 9
    30: {
        "gold_is_requirement": True,
        "gold_type": "education",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "Electrical Engineering", "type": "education_field"},
            {"name": "Engineering Management", "type": "education_field"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 31
    31: {
        "gold_is_requirement": True,
        "gold_type": "experience",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Electrical Project Management", "type": "domain_experience"},
            {"name": "Electrical Engineering", "type": "domain_experience"},
        ],
        "gold_min_years": 10,
        "gold_max_years": None,
    },

    # 33
    33: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Multi-project Management", "type": "capability"},
            {"name": "Task Prioritization", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 35 — JD 10
    35: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "required",
        "gold_components": [
            {"name": "MATLAB", "type": "tool"},
            {"name": "HelioScope", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 36
    36: {
        "gold_is_requirement": True,
        "gold_type": "education",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Electrical Engineering", "type": "education_field"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 38 — JD 11
    38: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Data Analysis", "type": "technical_skill"},
            {"name": "Power BI", "type": "tool"},
            {"name": "Excel", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 39
    39: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Leadership", "type": "soft_skill"},
            {"name": "Complex Project Management", "type": "capability"},
            {"name": "Team Management", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 40 — JD 12
    40: {
        "gold_is_requirement": True,
        "gold_type": "domain_knowledge",
        "gold_importance": "required",
        "gold_components": [
            {"name": "B2B Sales", "type": "domain_knowledge"},
            {"name": "Marketing", "type": "technical_skill"},
            {"name": "IT", "type": "domain_knowledge"},
            {"name": "Manufacturing", "type": "domain_knowledge"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 41
    41: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Market Research", "type": "domain_knowledge"},
            {"name": "Data Analysis", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 44 — JD 13
    44: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "preferred",
        "gold_components": [
            {"name": "GA4", "type": "tool"},
            {"name": "Power BI", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 45 — JD 14
    45: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Analytical Skills", "type": "capability"},
            {"name": "Project Management", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 46
    46: {
        "gold_is_requirement": True,
        "gold_type": "domain_knowledge",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Digital Marketing", "type": "technical_skill"},
            {"name": "Digital Marketing Channels", "type": "domain_knowledge"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 49
    49: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "SEO", "type": "technical_skill"},
            {"name": "SEM", "type": "technical_skill"},
            {"name": "Google Analytics", "type": "tool"},
            {"name": "CRM Software", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 51 — JD 15
    51: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "MATLAB", "type": "tool"},
            {"name": "AutoCAD", "type": "tool"},
            {"name": "Proteus", "type": "tool"},
            {"name": "Arduino IDE", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 53
    53: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Project Management Tools", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 54
    54: {
        "gold_is_requirement": True,
        "gold_type": "education",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Electrical Engineering", "type": "education_field"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 56 — JD 16
    56: {
        "gold_is_requirement": True,
        "gold_type": "experience",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Wireless Engineering", "type": "domain_experience"},
            {"name": "Telecommunications Engineering", "type": "domain_experience"},
        ],
        "gold_min_years": 8,
        "gold_max_years": None,
    },

    # 57
    57: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "required",
        "gold_components": [
            {"name": "GSM", "type": "technical_skill"},
            {"name": "UMTS", "type": "technical_skill"},
            {"name": "LTE", "type": "technical_skill"},
            {"name": "NR", "type": "technical_skill"},
            {"name": "Network Troubleshooting", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 60
    60: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Huawei RAN-SRAN", "type": "tool"},
            {"name": "Nokia RAN-SRAN", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 61
    61: {
        "gold_is_requirement": True,
        "gold_type": "soft_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Leadership", "type": "soft_skill"},
            {"name": "Communication", "type": "soft_skill"},
            {"name": "Project Management", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 63
    63: {
        "gold_is_requirement": True,
        "gold_type": "capability",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Independent Work", "type": "capability"},
            {"name": "Multi-project Management", "type": "capability"},
            {"name": "Deadline Management", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 66 — JD 17
    66: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Electronic Circuit Design", "type": "technical_skill"},
            {"name": "PCB Design", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 67
    67: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Embedded Systems", "type": "technical_skill"},
            {"name": "Embedded Systems Applications", "type": "domain_knowledge"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 69
    69: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Engineering Software", "type": "tool"},
            {"name": "Engineering Methods", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 75 — NOTE: these IDs correspond to selected example IDs,
    # not the row number in the final 50-row table.
    75: {
        "gold_is_requirement": True,
        "gold_type": "soft_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Communication", "type": "soft_skill"},
            {"name": "Teamwork", "type": "soft_skill"},
            {"name": "Report Writing", "type": "capability"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 76
    76: {
        "gold_is_requirement": True,
        "gold_type": "experience",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Automation Projects", "type": "domain_experience"},
            {"name": "Mobility Technologies", "type": "domain_knowledge"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 81 — JD 16
    81: {
        "gold_is_requirement": True,
        "gold_type": "education",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Telecommunications Engineering", "type": "education_field"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 82
    82: {
        "gold_is_requirement": True,
        "gold_type": "experience",
        "gold_importance": "required",
        "gold_components": [
            {"name": "Wireless Engineering", "type": "domain_experience"},
            {"name": "Telecommunications Engineering", "type": "domain_experience"},
        ],
        "gold_min_years": 8,
        "gold_max_years": None,
    },

    # 84
    84: {
        "gold_is_requirement": True,
        "gold_type": "tool",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Huawei RAN-SRAN Equipment", "type": "tool"},
            {"name": "Nokia RAN-SRAN Equipment", "type": "tool"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 90 — JD 17
    90: {
        "gold_is_requirement": True,
        "gold_type": "technical_skill",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Electronic Circuit Design", "type": "technical_skill"},
            {"name": "PCB Design", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },

    # 92
    92: {
        "gold_is_requirement": True,
        "gold_type": "domain_knowledge",
        "gold_importance": "unspecified",
        "gold_components": [
            {"name": "Embedded Systems", "type": "technical_skill"},
        ],
        "gold_min_years": None,
        "gold_max_years": None,
    },
})

In [116]:
missing_ids = sorted(
    set(selected_ids) - set(gold_annotations.keys())
)

extra_ids = sorted(
    set(gold_annotations.keys()) - set(selected_ids)
)

print("Selected examples:", len(selected_ids))
print("Annotated examples:", len(gold_annotations))
print("Missing IDs:", missing_ids)
print("Extra IDs:", extra_ids)

Selected examples: 50
Annotated examples: 61
Missing IDs: []
Extra IDs: [3, 4, 7, 10, 11, 16, 17, 20, 22, 28, 29]


In [117]:
gold_annotations = {
    example_id: annotation
    for example_id, annotation in gold_annotations.items()
    if example_id in selected_ids
}

print("Selected examples:", len(selected_ids))
print("Annotated examples:", len(gold_annotations))

missing_ids = sorted(
    set(selected_ids) - set(gold_annotations.keys())
)

extra_ids = sorted(
    set(gold_annotations.keys()) - set(selected_ids)
)

print("Missing IDs:", missing_ids)
print("Extra IDs:", extra_ids)

Selected examples: 50
Annotated examples: 50
Missing IDs: []
Extra IDs: []


In [120]:
# Rebuild the 50-row Gold table from scratch
gold_pilot = (
    candidate_requirements_df[
        candidate_requirements_df["example_id"].isin(selected_ids)
    ][["example_id", "jd_id", "text"]]
    .sort_values("example_id")
    .reset_index(drop=True)
)

# Add empty annotation columns
gold_pilot["gold_is_requirement"] = None
gold_pilot["gold_type"] = None
gold_pilot["gold_importance"] = None
gold_pilot["gold_components"] = None
gold_pilot["gold_min_years"] = None
gold_pilot["gold_max_years"] = None

# Apply annotations safely, one cell at a time
for example_id, annotation in gold_annotations.items():
    row_idx = gold_pilot.index[
        gold_pilot["example_id"] == example_id
    ]

    if len(row_idx) == 0:
        continue

    row_idx = row_idx[0]

    for column, value in annotation.items():
        gold_pilot.at[row_idx, column] = value

print("Gold rows:", len(gold_pilot))
print(
    "Annotated rows:",
    gold_pilot["gold_is_requirement"].notna().sum()
)

Gold rows: 50
Annotated rows: 50


In [121]:
print("Rows:", len(gold_pilot))
print("Unique IDs:", gold_pilot["example_id"].nunique())

print("\nMissing annotations:")
print(
    gold_pilot[
        gold_pilot["gold_is_requirement"].isna()
    ][["example_id", "text"]]
)

print("\nRequirement labels:")
print(
    gold_pilot["gold_is_requirement"].value_counts()
)

print("\nRequirement types:")
print(
    gold_pilot.loc[
        gold_pilot["gold_is_requirement"] == True,
        "gold_type"
    ].value_counts()
)

print("\nImportance:")
print(
    gold_pilot.loc[
        gold_pilot["gold_is_requirement"] == True,
        "gold_importance"
    ].value_counts()
)

Rows: 50
Unique IDs: 50

Missing annotations:
Empty DataFrame
Columns: [example_id, text]
Index: []

Requirement labels:
gold_is_requirement
True     48
False     2
Name: count, dtype: int64

Requirement types:
gold_type
technical_skill     11
tool                10
education            6
experience           6
soft_skill           5
domain_knowledge     5
capability           4
responsibility       1
Name: count, dtype: int64

Importance:
gold_importance
unspecified    24
required       18
preferred       6
Name: count, dtype: int64


In [122]:
gold_pilot["baseline_is_requirement"] = (
    gold_pilot["text"].apply(is_requirement)
)

gold_pilot["baseline_type"] = (
    gold_pilot["text"].apply(classify_requirement_type)
)

gold_pilot["baseline_importance"] = (
    gold_pilot["text"].apply(detect_importance)
)

print(
    gold_pilot[
        [
            "example_id",
            "gold_is_requirement",
            "baseline_is_requirement",
            "gold_type",
            "baseline_type",
        ]
    ].to_string(index=False)
)

 example_id gold_is_requirement  baseline_is_requirement        gold_type   baseline_type
          0                True                     True        education       education
          1                True                     True       experience      experience
          2                True                     True             tool technical_skill
          5                True                     True  technical_skill      experience
          6                True                     True       soft_skill           other
          8                True                     True domain_knowledge      experience
          9                True                     True       soft_skill      soft_skill
         12                True                     True       soft_skill           other
         13                True                     True             tool           other
         14                True                     True             tool      experience
         1

In [123]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

y_true = gold_pilot["gold_is_requirement"].astype(bool)
y_pred = gold_pilot["baseline_is_requirement"].astype(bool)

print("Requirement Detection")
print("---------------------")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1       :", f1_score(y_true, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Requirement Detection
---------------------
Accuracy : 0.96
Precision: 0.96
Recall   : 1.0
F1       : 0.9795918367346939

Confusion Matrix:
[[ 0  2]
 [ 0 48]]


In [124]:
errors = gold_pilot[
    gold_pilot["gold_is_requirement"]
    != gold_pilot["baseline_is_requirement"]
][
    [
        "example_id",
        "jd_id",
        "text",
        "gold_is_requirement",
        "baseline_is_requirement",
    ]
]

print("Number of errors:", len(errors))
errors

Number of errors: 2


,example_id,jd_id,text,gold_is_requirement,baseline_is_requirement
10,15,2,Advanced knowledge of CRM software and digital marketing tools.,False,True
13,21,3,Proficiency in CRM software for managing client relationships.,False,True


In [125]:
type_eval = gold_pilot[
    gold_pilot["gold_is_requirement"] == True
].copy()

type_accuracy = accuracy_score(
    type_eval["gold_type"],
    type_eval["baseline_type"]
)

print("Requirement Type Accuracy:", type_accuracy)

print("\nType Confusion Matrix:")
print(
    pd.crosstab(
        type_eval["gold_type"],
        type_eval["baseline_type"],
        rownames=["Gold"],
        colnames=["Baseline"],
        dropna=False
    )
)

Requirement Type Accuracy: 0.125

Type Confusion Matrix:
Baseline          education  experience  other  soft_skill  technical_skill
Gold                                                                       
capability                0           2      1           0                1
domain_knowledge          0           1      3           1                0
education                 3           2      1           0                0
experience                0           2      0           3                1
responsibility            0           0      1           0                0
soft_skill                0           0      3           1                1
technical_skill           0           2      7           2                0
tool                      2           3      4           0                1


In [126]:
print("Gold pilot:")
print(
    gold_pilot[
        [
            "example_id",
            "jd_id",
            "text",
            "gold_is_requirement",
            "baseline_is_requirement",
            "gold_type",
            "baseline_type",
        ]
    ].to_string(index=False)
)

Gold pilot:
 example_id  jd_id                                                                                                                                    text gold_is_requirement  baseline_is_requirement        gold_type   baseline_type
          0      0                                                              Bachelor’s degree in Business Administration, Economics, or related field.                True                     True        education       education
          1      0                                                                      3-5 years of experience in business development or related fields.                True                     True       experience      experience
          2      0                                                             Strong proficiency with MS Office Suite and data analysis tools like Stata.                True                     True             tool technical_skill
          5      1                                      

In [127]:
errors = gold_pilot[
    gold_pilot["gold_is_requirement"]
    != gold_pilot["baseline_is_requirement"]
]

print("Errors:", len(errors))
print(
    errors[
        [
            "example_id",
            "jd_id",
            "text",
            "gold_is_requirement",
            "baseline_is_requirement",
        ]
    ].to_string(index=False)
)

Errors: 2
 example_id  jd_id                                                            text gold_is_requirement  baseline_is_requirement
         15      2 Advanced knowledge of CRM software and digital marketing tools.               False                     True
         21      3  Proficiency in CRM software for managing client relationships.               False                     True


In [128]:
gold_pilot[
    ["example_id", "jd_id", "text"]
].to_string(index=False)

" example_id  jd_id                                                                                                                                    text\n          0      0                                                              Bachelor’s degree in Business Administration, Economics, or related field.\n          1      0                                                                      3-5 years of experience in business development or related fields.\n          2      0                                                             Strong proficiency with MS Office Suite and data analysis tools like Stata.\n          5      1                                                                                                      ### Desired Skills and Experience:\n          6      1                                                                                        Strong programming skills in C++, HTML, and CSS.\n          8      1                                             

In [129]:
# Reset only the Gold annotation columns
gold_columns = [
    "gold_is_requirement",
    "gold_type",
    "gold_importance",
    "gold_components",
    "gold_min_years",
    "gold_max_years",
]

for col in gold_columns:
    gold_pilot[col] = None

In [130]:
def make_annotation(
    is_requirement,
    gold_type="other",
    importance="unspecified",
    components=None,
    min_years=None,
    max_years=None
):
    return {
        "gold_is_requirement": is_requirement,
        "gold_type": gold_type,
        "gold_importance": importance,
        "gold_components": components or [],
        "gold_min_years": min_years,
        "gold_max_years": max_years,
    }


def component(name, component_type):
    return {
        "name": name,
        "type": component_type
    }


gold_annotations = {

    # --------------------------------------------------
    # JD 0
    # --------------------------------------------------

    0: make_annotation(
        True,
        "education",
        "unspecified",
        [
            component("Bachelor's", "degree"),
            component("Business Administration", "education_field"),
            component("Economics", "education_field"),
        ]
    ),

    1: make_annotation(
        True,
        "experience",
        "unspecified",
        [
            component("Business Development", "domain_experience")
        ],
        min_years=3,
        max_years=5
    ),

    2: make_annotation(
        True,
        "tool",
        "unspecified",
        [
            component("MS Office Suite", "tool"),
            component("Data Analysis", "technical_skill"),
            component("Stata", "tool"),
        ]
    ),

    # --------------------------------------------------
    # JD 1
    # --------------------------------------------------

    5: make_annotation(
        False
    ),

    6: make_annotation(
        True,
        "technical_skill",
        "preferred",
        [
            component("C++", "programming_language"),
            component("HTML", "markup_language"),
            component("CSS", "stylesheet_language"),
        ]
    ),

    8: make_annotation(
        True,
        "experience",
        "preferred",
        [
            component("Large Dataset Management", "technical_skill"),
            component("Power BI", "tool"),
            component("Excel", "tool"),
            component("Tableau", "tool"),
        ]
    ),

    9: make_annotation(
        True,
        "soft_skill",
        "preferred",
        [
            component("Communication", "soft_skill"),
            component("Client Relationship Development", "soft_skill"),
        ]
    ),

    # --------------------------------------------------
    # JD 2
    # --------------------------------------------------

    12: make_annotation(
        False
    ),

    13: make_annotation(
        True,
        "capability",
        "preferred",
        [
            component("Strategic Planning", "capability"),
            component("Market Analysis", "capability"),
        ]
    ),

    14: make_annotation(
        True,
        "soft_skill",
        "preferred",
        [
            component("Leadership", "soft_skill"),
            component("Team Development", "capability"),
        ]
    ),

    15: make_annotation(
        True,
        "tool",
        "preferred",
        [
            component("CRM Software", "tool"),
            component("Digital Marketing Tools", "tool"),
        ]
    ),

    # --------------------------------------------------
    # JD 3
    # --------------------------------------------------

    18: make_annotation(
        False
    ),

    19: make_annotation(
        True,
        "soft_skill",
        "unspecified",
        [
            component("Communication", "soft_skill"),
            component("Interpersonal Skills", "soft_skill"),
            component("Client Engagement", "capability"),
            component("Collaboration", "soft_skill"),
        ]
    ),

    21: make_annotation(
        True,
        "tool",
        "unspecified",
        [
            component("CRM Software", "tool"),
            component("Client Relationship Management", "capability"),
        ]
    ),

    # --------------------------------------------------
    # JD 4
    # --------------------------------------------------

    23: make_annotation(
        False
    ),

    24: make_annotation(
        True,
        "experience",
        "required",
        [
            component("Sales Management", "domain_experience"),
            component("Digital Marketing", "domain_experience"),
            component("5+ Years", "experience_threshold_preferred"),
        ],
        min_years=5
    ),

    25: make_annotation(
        True,
        "capability",
        "required",
        [
            component("Strategic Goal Setting", "capability"),
            component("Problem Solving", "soft_skill"),
            component("Communication", "soft_skill"),
        ]
    ),

    26: make_annotation(
        True,
        "responsibility",
        "required",
        [
            component("Team Leadership", "responsibility"),
            component("Team Supervision", "responsibility"),
        ]
    ),

    # --------------------------------------------------
    # JD 5
    # --------------------------------------------------

    27: make_annotation(
        True,
        "technical_skill",
        "unspecified",
        [
            component("Sales", "domain_knowledge"),
            component("Marketing", "domain_knowledge"),
            component("Analytical Skills", "capability"),
            component("SEO", "technical_skill"),
        ]
    ),

    # --------------------------------------------------
    # JD 6
    # --------------------------------------------------

    30: make_annotation(
        False
    ),

    31: make_annotation(
        True,
        "soft_skill",
        "required",
        [
            component("Leadership", "soft_skill"),
            component("Stakeholder Management", "capability"),
        ]
    ),

    33: make_annotation(
        True,
        "tool",
        "required",
        [
            component("Salesforce", "tool"),
            component("Vistex", "tool"),
            component("SAP ERP", "tool"),
        ]
    ),

    35: make_annotation(
        True,
        "education",
        "required",
        [
            {
                "name": "Master's",
                "type": "degree",
                "importance": "preferred"
            },
            {
                "name": "Bachelor's",
                "type": "degree",
                "importance": "required"
            },
            {
                "name": "Business Administration",
                "type": "education_field",
                "importance": "required"
            }
        ]
    ),

    36: make_annotation(
        True,
        "experience",
        "required",
        [
            component("Sales Management", "domain_experience"),
            component("Business Development", "domain_experience"),
            component("Key Account Management", "domain_experience"),
        ],
        min_years=5
    ),

    # --------------------------------------------------
    # JD 7
    # --------------------------------------------------

    38: make_annotation(
        True,
        "tool",
        "preferred",
        [
            component("Power BI", "tool"),
            component("Oracle DV", "tool"),
            component("Tableau", "tool"),
        ]
    ),

    39: make_annotation(
        True,
        "domain_knowledge",
        "preferred",
        [
            component("Agile", "methodology"),
            component("SCRUM", "methodology"),
            component("Software Development Methodologies", "domain_knowledge"),
        ]
    ),

    40: make_annotation(
        True,
        "capability",
        "preferred",
        [
            component("Technical Team Leadership", "capability"),
            component("Stakeholder Communication", "soft_skill"),
        ]
    ),

    # --------------------------------------------------
    # JD 8
    # --------------------------------------------------

    41: make_annotation(
        True,
        "capability",
        "unspecified",
        [
            component("Analytical Skills", "capability"),
            component("Opportunity Identification", "capability"),
            component("Market Trend Analysis", "capability"),
        ]
    ),

    # --------------------------------------------------
    # JD 9
    # --------------------------------------------------

    44: make_annotation(
        True,
        "education",
        "required",
        [
            {
                "name": "Bachelor's",
                "type": "degree",
                "importance": "required"
            },
            {
                "name": "Electrical Engineering",
                "type": "education_field",
                "importance": "required"
            },
            {
                "name": "Master's",
                "type": "degree",
                "importance": "preferred"
            },
            {
                "name": "Engineering Management",
                "type": "education_field",
                "importance": "preferred"
            }
        ]
    ),

    45: make_annotation(
        True,
        "experience",
        "unspecified",
        [
            component("Electrical Project Management", "domain_experience"),
            component("Electrical Engineering", "domain_experience"),
        ],
        min_years=10
    ),

    46: make_annotation(
        True,
        "technical_skill",
        "unspecified",
        [
            component("Electrical Design Evaluation", "technical_skill"),
            component("Project Management", "capability"),
            component("Team Leadership", "capability"),
        ]
    ),

    49: make_annotation(
        True,
        "capability",
        "unspecified",
        [
            component("Multi-Project Management", "capability"),
            component("Task Prioritization", "capability"),
        ]
    ),

    # --------------------------------------------------
    # JD 10
    # --------------------------------------------------

    51: make_annotation(
        True,
        "tool",
        "unspecified",
        [
            component("MATLAB", "tool"),
            component("HelioScope", "tool"),
            component("Solar System Design", "domain_knowledge"),
        ]
    ),

    53: make_annotation(
        True,
        "tool",
        "unspecified",
        [
            component("Project Management Tools", "tool")
        ]
    ),

    54: make_annotation(
        True,
        "education",
        "unspecified",
        [
            component("Bachelor's", "degree"),
            component("Electrical Engineering", "education_field"),
        ]
    ),

    # --------------------------------------------------
    # JD 11
    # --------------------------------------------------

    56: make_annotation(
        True,
        "technical_skill",
        "unspecified",
        [
            component("Analytical Skills", "capability"),
            component("Data Analysis", "technical_skill"),
            component("Power BI", "tool"),
            component("Excel", "tool"),
        ]
    ),

    57: make_annotation(
        True,
        "capability",
        "unspecified",
        [
            component("Leadership", "soft_skill"),
            component("Complex Project Management", "capability"),
            component("Team Management", "capability"),
        ]
    ),

    # --------------------------------------------------
    # JD 12
    # --------------------------------------------------

    60: make_annotation(
        True,
        "experience",
        "required",
        [
            component("B2B Sales", "domain_experience"),
            component("Marketing", "domain_experience"),
            component("IT", "domain_knowledge"),
            component("Manufacturing", "domain_knowledge"),
        ]
    ),

    61: make_annotation(
        True,
        "technical_skill",
        "required",
        [
            component("Market Research", "technical_skill"),
            component("Data Analysis", "technical_skill"),
        ]
    ),

    # --------------------------------------------------
    # JD 13
    # --------------------------------------------------

    63: make_annotation(
        True,
        "tool",
        "preferred",
        [
            component("GA4", "tool"),
            component("Power BI", "tool"),
        ]
    ),

    # --------------------------------------------------
    # JD 14
    # --------------------------------------------------

    66: make_annotation(
        True,
        "capability",
        "unspecified",
        [
            component("Analytical Skills", "capability"),
            component("Project Management", "capability"),
        ]
    ),

    67: make_annotation(
        True,
        "domain_knowledge",
        "unspecified",
        [
            component("Digital Marketing Channels", "domain_knowledge")
        ]
    ),

    69: make_annotation(
        True,
        "technical_skill",
        "unspecified",
        [
            component("SEO", "technical_skill"),
            component("SEM", "technical_skill"),
            component("Google Analytics", "tool"),
            component("CRM Software", "tool"),
        ]
    ),

    # --------------------------------------------------
    # JD 15
    # --------------------------------------------------

    75: make_annotation(
        True,
        "tool",
        "unspecified",
        [
            component("MATLAB", "tool"),
            component("AutoCAD", "tool"),
            component("Proteus", "tool"),
            component("Arduino IDE", "tool"),
        ]
    ),

    76: make_annotation(
        True,
        "soft_skill",
        "unspecified",
        [
            component("Communication", "soft_skill"),
            component("Teamwork", "soft_skill"),
            component("Report Writing", "soft_skill"),
        ]
    ),

    # --------------------------------------------------
    # JD 16
    # --------------------------------------------------

    81: make_annotation(
        True,
        "education",
        "unspecified",
        [
            component("Bachelor's", "degree"),
            component("Telecommunications Engineering", "education_field"),
        ]
    ),

    82: make_annotation(
        True,
        "experience",
        "unspecified",
        [
            component("Wireless Engineering", "domain_experience"),
            component("Telecommunications Engineering", "domain_experience"),
        ],
        min_years=8
    ),

    84: make_annotation(
        True,
        "tool",
        "unspecified",
        [
            component("Huawei RAN-SRAN", "tool"),
            component("Nokia RAN-SRAN", "tool"),
        ]
    ),

    # --------------------------------------------------
    # JD 17
    # --------------------------------------------------

    90: make_annotation(
        True,
        "technical_skill",
        "unspecified",
        [
            component("Electronic Circuit Design", "technical_skill"),
            component("PCB Design", "technical_skill"),
        ]
    ),

    92: make_annotation(
        True,
        "technical_skill",
        "unspecified",
        [
            component("Embedded Systems", "technical_skill"),
            component("Embedded Applications", "technical_skill"),
        ]
    ),
}

In [131]:
for example_id, annotation in gold_annotations.items():

    row_idx = gold_pilot.index[
        gold_pilot["example_id"] == example_id
    ]

    if len(row_idx) != 1:
        print("WARNING:", example_id, "matched", len(row_idx), "rows")
        continue

    row_idx = row_idx[0]

    for column, value in annotation.items():
        gold_pilot.at[row_idx, column] = value

In [132]:
print("Gold annotations:", len(gold_annotations))
print("Gold pilot rows:", len(gold_pilot))

print("\nMissing annotations:")
print(
    gold_pilot[
        gold_pilot["gold_is_requirement"].isna()
    ][["example_id", "text"]]
)

print("\nRequirement count:")
print(gold_pilot["gold_is_requirement"].value_counts())

print("\nFalse rows:")
display(
    gold_pilot[
        gold_pilot["gold_is_requirement"] == False
    ][["example_id", "jd_id", "text"]]
)

print("\nTrue rows:")
display(
    gold_pilot[
        gold_pilot["gold_is_requirement"] == True
    ][["example_id", "jd_id", "text"]]
)

Gold annotations: 50
Gold pilot rows: 50

Missing annotations:
Empty DataFrame
Columns: [example_id, text]
Index: []

Requirement count:
gold_is_requirement
True     45
False     5
Name: count, dtype: int64

False rows:


,example_id,jd_id,text
3,5,1,### Desired Skills and Experience:
7,12,2,**Preferred Skills:**
11,18,3,#### Skills and Competencies:
14,23,4,### Essential Skills and Experience
19,30,6,#### Essential Skills:



True rows:


,example_id,jd_id,text
0,0,0,"Bachelor’s degree in Business Administration, Economics, or related field."
1,1,0,3-5 years of experience in business development or related fields.
2,2,0,Strong proficiency with MS Office Suite and data analysis tools like Stata.
4,6,1,"Strong programming skills in C++, HTML, and CSS."
5,8,1,"Experience in managing large data sets and using BI tools (Power BI, Excel, Tableau)."
6,9,1,Excellent communication skills and ability to develop strong client relationships.
8,13,2,Proficiency in strategic planning and market analysis.
9,14,2,Strong leadership skills with proven experience in team development.
10,15,2,Advanced knowledge of CRM software and digital marketing tools.
12,19,3,Strong communication and interpersonal skills for client engagement and collaboration.


In [133]:
gold_pilot["baseline_is_requirement"] = (
    gold_pilot["text"].apply(is_requirement)
)

gold_pilot["baseline_type"] = (
    gold_pilot["text"].apply(classify_requirement_type)
)

gold_pilot["baseline_importance"] = (
    gold_pilot["text"].apply(detect_importance)
)

In [134]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = gold_pilot["gold_is_requirement"].astype(bool)
y_pred = gold_pilot["baseline_is_requirement"].astype(bool)

print("Requirement Detection")
print("---------------------")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1       :", f1_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Requirement Detection
---------------------
Accuracy : 0.9
Precision: 0.9
Recall   : 1.0
F1       : 0.9473684210526315

Confusion Matrix:
[[ 0  5]
 [ 0 45]]


In [135]:
def is_section_heading(text):
    if not isinstance(text, str):
        return False

    text = text.strip()

    # Markdown headings
    if re.match(r"^#{1,6}\s+", text):
        return True

    # Bold-only headings
    if re.match(r"^\*\*.*\*\*:?\s*$", text):
        return True

    return False

In [136]:
def is_requirement(text):
    if is_section_heading(text):
        return False

    text_lower = text.lower()

    return any(
        re.search(pattern, text_lower)
        for pattern in REQUIREMENT_PATTERNS
    )

In [137]:
gold_pilot["baseline_is_requirement_v2"] = (
    gold_pilot["text"].apply(is_requirement)
)

y_true = gold_pilot["gold_is_requirement"].astype(bool)
y_pred = gold_pilot["baseline_is_requirement_v2"].astype(bool)

print("Requirement Detection — V2")
print("--------------------------")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1       :", f1_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Requirement Detection — V2
--------------------------
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1       : 1.0

Confusion Matrix:
[[ 5  0]
 [ 0 45]]


In [138]:
errors = gold_pilot[
    gold_pilot["gold_is_requirement"] !=
    gold_pilot["baseline_is_requirement"]
]

display(
    errors[
        [
            "example_id",
            "jd_id",
            "text",
            "gold_is_requirement",
            "baseline_is_requirement"
        ]
    ]
)

,example_id,jd_id,text,gold_is_requirement,baseline_is_requirement
3,5,1,### Desired Skills and Experience:,False,True
7,12,2,**Preferred Skills:**,False,True
11,18,3,#### Skills and Competencies:,False,True
14,23,4,### Essential Skills and Experience,False,True
19,30,6,#### Essential Skills:,False,True


In [139]:
errors_v2 = gold_pilot[
    gold_pilot["gold_is_requirement"] !=
    gold_pilot["baseline_is_requirement_v2"]
]

display(
    errors_v2[
        [
            "example_id",
            "jd_id",
            "text",
            "gold_is_requirement",
            "baseline_is_requirement_v2"
        ]
    ]
)

,example_id,jd_id,text,gold_is_requirement,baseline_is_requirement_v2


In [141]:
errors_v2 = gold_pilot[
    gold_pilot["gold_is_requirement"] !=
    gold_pilot["baseline_is_requirement_v2"]
]

display(
    errors_v2[
        [
            "example_id",
            "jd_id",
            "text",
            "gold_is_requirement",
            "baseline_is_requirement_v2"
        ]
    ]
)

,example_id,jd_id,text,gold_is_requirement,baseline_is_requirement_v2


## Baseline Results

### Requirement Detection

| Version | Approach | Accuracy | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| V1 | Lexical rule-based baseline | 90.0% | 90.0% | 100.0% | 94.74% |
| V2 | Baseline + structural heading detection | 100.0% | 100.0% | 100.0% | 100.0% |

### Error Analysis

The V1 baseline produced five false positives. All five were section
headings containing words such as "skills" or "experience".

The error analysis motivated a structural heading detector. V2 filters
section headings before applying lexical requirement patterns and removed
all five false positives on the 50-example Gold pilot.

### Limitation

The Gold pilot contains only 50 manually validated examples and is not
sufficient to claim that requirement extraction is solved. The results
represent a baseline experiment and motivate evaluation on a larger,
more diverse benchmark.

### Next Step

Evaluate requirement-type classification across education, experience,
tools, technical skills, soft skills, capabilities, responsibilities,
and other requirement categories.